### Ripple-Carry Adder in SuperNeuroMAT

**Tutorial Author:** Simon Weston <br> 
**Algorithmic Design:** Simon Weston, Duncan Rowden, Prasanna Date <br>

In this tutorial we'll create a generalized spiking neural network for adding two numbers. The system takes in two n-digit binary inputs and produces a binary output, using spikes to represent 1's and the absence of spikes to represent 0's. This network models the add-and-carry mechanism of binary addition through synaptic connections and threshold logic.

#### System Structure

![ripple-carry adder system diagram](img/ripple_carry_adder_images/ripple_carry_adder_diagram.png)

In the two-bit system above, thresholds are denoted inside neurons, and synapses have parameters <weight, delay>

The x and y bit neurons are manually spiked according to the inputs' binary representation, the 1's propogate the bit clusters, then finish by spiking z, the binary output.

Bits - red neuron clusters: each bit is represented by a cluster of 3 red neurons, all receiving the same input. They each take in all spikes from the corresponding input bits and the carry over of the preceding bit. The least significant bit cluster only has 2 red neurons because there isn't a possible carry there.



By setting thresholds to 0, 1, and 2 for each neuron in the bit clusters, they detect and respond accordingly to differing amounts of spikes:<br>

![middle layer table](img/ripple_carry_adder_images/ripple_carry_adder_logic_table.png)
Cluster receives 0 spikes: no cluster neurons spike, output bit remains 0 <br>
Cluster receives 1 spike: only thres=0 spikes, output bit spikes, denoting 1 <br>
Cluster receives 2 spikes: both thres=0 and thres=1 spike, triggering the carry mechanism and negating the output bit, denoting 0 <br>
Cluster receives 3 spikes: all three thresholds spike, both triggering the carry mechanism and summing (1-1+1), output bit spikes, denoting 1 <br>

Delays: carry over signals need to arrive at the same time as input spikes, so their synapses have offset delays (bit + 1). Each cluster bit finishes its calculation right before the one above it does, so the cluster-to-output synapses are offset as well.

#### Step 1: Set Up and Creating Neurons 

In [74]:
#If you haven't yet installed SuperNeuroMat, copy the following and run it in your terminal: 
#pip install superneuromat

# Let's begin by importing the SuperNeuroMAT library and naming our algorithm:
import superneuromat as snm
ADD = snm.SNN()

Our input neurons have threshold = 0 (we will manually spike them):

In [75]:
bit_count = 2         # (placeholder, we will move to a test section under the network creation section later)

x_input = []
y_input = []

# create inputs
for i in range(bit_count):
    x_input.append(ADD.create_neuron(threshold=0))

for i in range(bit_count):
    y_input.append(ADD.create_neuron(threshold=0))

Now we'll create the bit clusters (red neurons). Using a loop, we'll group each neuron into clusters, then nest our clusters inside a larger bit_groups list.  

In [76]:
bit_groups = []

# create hidden layer

for bit in range(bit_count):

    cluster = []

    if bit == 0:
        thresholds = [0, 1]
    else:
        thresholds = [0, 1, 2]

    for threshold in thresholds:
        neuron = ADD.create_neuron(threshold=threshold)
        cluster.append(neuron)
        bit_groups.append(cluster)



Finally, let's create output neurons. There are bit_count + 1 outputs because there's a possibility of having a final bit carry over from our highest digit place.

In [77]:
z_outputs = []

# create output neurons

for i in range(bit_count + 1):
    z_outputs.append(ADD.create_neuron(threshold=0))


#### Step 2: Create Synapses

First, our input to hidden layer synapses connect every input bit with all respective cluster neurons. Delay has a +1 increase for each bit because we need to account for carry overs, ensuring that all spikes arrive to each cluster at the same timestep.

In [78]:
# input to hidden layer synapses

for bit in range(bit_count):
    
    delay = bit + 1
    
    for neuron in bit_groups[bit]:
        ADD.create_synapse(
                pre_id = x_input[bit],
                post_id = neuron,
                weight = 1.0,
                delay = delay
            )
        ADD.create_synapse(
                pre_id = y_input[bit],
                post_id = neuron,
                weight = 1.0,
                delay = delay
            )

Next, to allow bits to carry over when a bit place = 2, we connect each hidden layer neuron with threshold = 1 to all neurons of the subsequent bit group.

In [79]:
# carry over synapses between bit groups

# applies to all but the highest bit (see network diagram)
for bit in range(bit_count - 1):

# calling 2nd neuron of each bit (thres = 1):
    carry_neuron = bit_groups[bit][1]

# link carry_neuron to all neurons in the next highest bit group (bit + 1)
    for neuron in bit_groups[bit + 1]:
        ADD.create_synapse(
            carry_neuron,
            neuron,
            weight = 1.0,
            delay = 1
        )

Now for the hidden layer to output layer synapses. As discussed earlier, each neuron within a cluster serves a unique role in transmitting the correct spike amount to the output neurons, and the weights of these synapses are unique to each neuron's threshold. Additionally, the delays for each bit group are staggered to transmit all output results at the same time step.

In [80]:
# hidden to output layer synapses
        
for bit in range(bit_count):
    
    delay = bit_count - bit
            
    threshold_0 = bit_groups[bit][0]
    threshold_1 = bit_groups[bit][1]

# threshold = 0 to output

    ADD.create_synapse(
        pre_id = threshold_0,
        post_id = z_outputs[bit],
        weight = 1.0,
        delay = delay
    )

# threshold = 1 to output

    ADD.create_synapse(
        pre_id = threshold_1,
        post_id = z_outputs[bit],
        weight = -1.0,
        delay = delay
    )


# threshold = 2 to output  

# the 0 bit doesn't have a threshold = 2 because there's no carry over included in its sum
    if bit > 0:

        threshold_2 = cluster[2]

        ADD.create_synapse(
            pre_id = threshold_2,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )


As discussed earlier, our threshold 1 neuron of the highest bit won't carry over (there is no higher bit to carry to); instead it sends its final carry to the highest output bit:

In [81]:
# final carry to final output
final_carry = bit_groups[-1][1]

ADD.create_synapse(
    pre_id = final_carry,
    post_id = z_outputs[-1],
    weight = 1.0,
    delay = 1
    )

<Synapse 21 on SNN at 0x2ba49c8b0e0>

#### Step 3: Test (Add Spikes / Simulate)

To make data entry more convenient, let's organize what we currently have inside a function:

In [82]:

import superneuromat as snm

def create_ripple_carry_adder(bit_count):

    ADD = snm.SNN()

    x_input = []
    y_input = []
    bit_groups = []
    z_outputs = []

    # create inputs
    for i in range(bit_count):
        x_input.append(ADD.create_neuron(threshold=0))

    for i in range(bit_count):
        y_input.append(ADD.create_neuron(threshold=0))


    # create hidden layer
    for bit in range(bit_count):

        cluster = []

        if bit == 0:
            thresholds = [0, 1]
        else:
            thresholds = [0, 1, 2]

        for threshold in thresholds:
            neuron = ADD.create_neuron(threshold=threshold)
            cluster.append(neuron)

        bit_groups.append(cluster)

            

    # create output neurons
    for i in range(bit_count + 1):
        z_outputs.append(ADD.create_neuron(threshold=0))

    # input to hidden layer synapses
    for bit in range(bit_count):
        
        delay = bit + 1
        
        for neuron in bit_groups[bit]:
            ADD.create_synapse(
                    pre_id = x_input[bit],
                    post_id = neuron,
                    weight = 1.0,
                    delay = delay
                )
            ADD.create_synapse(
                    pre_id = y_input[bit],
                    post_id = neuron,
                    weight = 1.0,
                    delay = delay
                )
            
    # carry over synapses
    for bit in range(bit_count - 1):

        carry_neuron = bit_groups[bit][1]

        for neuron in bit_groups[bit + 1]:
            ADD.create_synapse(
                carry_neuron,
                neuron,
                weight = 1.0,
                delay = 1
            )

    # hidden to output layer synapses
    for bit in range(bit_count):
        
        delay = bit_count - bit
                
        threshold_0 = bit_groups[bit][0]
        threshold_1 = bit_groups[bit][1]

        ADD.create_synapse(
            pre_id = threshold_0,
            post_id = z_outputs[bit],
            weight = 1.0,
            delay = delay
        )

        ADD.create_synapse(
            pre_id = threshold_1,
            post_id = z_outputs[bit],
            weight = -1.0,
            delay = delay
        )

        if bit > 0:

            threshold_2 = bit_groups[bit][2]

            ADD.create_synapse(
                pre_id = threshold_2,
                post_id = z_outputs[bit],
                weight = 1.0,
                delay = delay
            )

    # final carry to final output
    final_carry = bit_groups[-1][1]

    ADD.create_synapse(
        pre_id = final_carry,
        post_id = z_outputs[-1],
        weight = 1.0,
        delay = 1
    )

    return ADD, x_input, y_input, z_outputs, bit_groups

Now let's create our test section. We convert inputted base-ten values to binary and add spikes accordingly. Then we simulate for bit_count + 2 timesteps.

In [83]:
BIT_COUNT = 2

ADD, x_input, y_input, z_outputs, bit_groups = create_ripple_carry_adder(BIT_COUNT)

x_value = 2
y_value = 1

for bit in range(BIT_COUNT):

    if (x_value >> bit) & 1:
             #  >> takes x_value's binary value, shifts bits to right _bit_ places, isolates last bit and returns true if == 1. (bitwise AND) 
        ADD.add_spike(0, x_input[bit])

    if (y_value >> bit) & 1:
        ADD.add_spike(0, y_input[bit])

# simulate
ADD.simulate(BIT_COUNT + 2)
 

#### Step 4: Results

We need an output that's easy to interpret. One way to do this is checking the spike train array for spikes on z_outputs and convert their binary to base ten for comparison with our expected value:

In [84]:
# decode output

output_bits = []

for neuron in z_outputs:
        
        neuron_id = neuron.idx

        spiked = any(
            ADD.spike_train[t][neuron_id]
            for t in range(len(ADD.spike_train))
        )

        if spiked:
            output_bits.append(1)
        else:
            output_bits.append(0)


# we need to reverse the order of our output bits for conventional binary since they're currently arranged from least to greatest 
binary_output = output_bits[::-1]

print("\nOutput bits:")
print(binary_output)

print("\nSpike train:")
ADD.print_spike_train()


# Convert binary to base-ten
result = 0

for bit in binary_output:
        result = result * 2 + bit

print("\nResult:", result)
print("Expected:", x_value + y_value)

if result == x_value + y_value:
        print("Correct")
else:
        print("Incorrect")


Output bits:
[0, 1, 1]

Spike train:
t|id0 1 2 3 4 5 6 7 8 9 10111213141516171819 
0: [│ ├─├─│ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ │ ]
1: [│ │ │ │ ├─│ │ │ │ │ │ │ ├─│ ├─│ ├─│ │ │ ]
2: [│ │ │ │ │ │ ├─│ │ │ │ │ │ │ │ │ │ │ ├─│ ]
3: [│ │ │ │ │ │ │ │ │ ├─├─│ │ │ │ │ │ │ │ │ ]

Result: 3
Expected: 3
Correct


Congratulations! You've completed this tutorial.